# Experimentation of CamelUp Code

In [13]:
%load_ext autoreload
%autoreload 2

from CamelUp import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Examination of structure of the field

In [14]:
demo_players = {"Player 1":player("Player 1"),"Player 2":player("Player 2"),
                "Player 3":player("Player 3"),"Player 4":player("Player 4")}

demoE = Field([
    [], [], [], [], [], ['Yellow', 'Green'], ['Purple', 'Blue', 'Orange'], ["DESERT", "Player 1"], [], 
    ["Black", "White"], [], [], [], [],[], [], [], [], []],copy.deepcopy(demo_players))


In [15]:
demoE.game_field
mask = {"Purple":1,"Blue":2,"Orange":3,"Yellow":4,"Green":5,"White":6,"Black":7,"DESERT":8,"OASIS":9}


In [16]:
rendered_field, inv_player_mask = render_field(demoE.game_field,demoE.players)
print(rendered_field)
print(inv_player_mask)

[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 4  5  0  0  0  0  0]
 [ 1  2  3  0  0  0  0]
 [ 8 10  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 7  6  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]]
{10: 'Player 1', 11: 'Player 2', 12: 'Player 3', 13: 'Player 4'}


### New generation flexible for testing
Step 1: create all permutations for the dice

In [ ]:
import itertools
camels_thrown = []
camels_not_thrown = [1,2,3,4,5,6]
positions = np.zeros((5,3))
DO_hits = np.zeros((len(demoE.players),1))
verbose = True

## use time to measure performance
if verbose:
    start_time = time.time()

len_all_camels = len(camels_thrown) + len(camels_not_thrown)
draw_n_camels = len(camels_not_thrown)
if len_all_camels > 5:
    draw_n_camels -= 1
print("draw_n_camels: ", draw_n_camels)
all_dice_permutations = list(itertools.product(range(1,4), repeat=draw_n_camels))
print("Number of paths: ", len(all_dice_permutations), "first 5 paths: ", all_dice_permutations[:5], "SHOULD number of paths", 3**draw_n_camels)
# Generate all possible permutations of camels_not_thrown (numba friendly: i.e., as lists of integer arrays)
# (This is separate from all_dice_permutations, which is all product/dice rolls)
all_camel_permutations = list(itertools.permutations(camels_not_thrown))
print("Number of permutations: ", len(all_camel_permutations), "first 5 permutations:", all_camel_permutations[:5], "should permutations")

## include white camel as well as black camel:
if 5 in camels_not_thrown:
    camels_not_thrown.remove(6)
    camels_not_thrown.append(7)
    all_camel_permutations.extend(list(itertools.permutations(camels_not_thrown)))

if verbose:
    interval_time_permutations = time.time()
    print("Time to generate permutations:", round(interval_time_permutations - start_time, 6), "seconds")

# Efficiently create all combinations of camel order (permutation) and dice rolls (product) using itertools.product
all_full_permutations = list(itertools.product(all_camel_permutations, all_dice_permutations))
print("Number of full permutations: ", len(all_full_permutations), "First 5:", all_full_permutations[:5], "should be", len(all_camel_permutations)*len(all_dice_permutations))

if verbose:
    interval_time_product = time.time()
    print("Time to generate product:", round(interval_time_product - interval_time_permutations, 6), "seconds")


draw_n_camels:  5
Number of paths:  243 first 5 paths:  [(1, 1, 1, 1, 1), (1, 1, 1, 1, 2), (1, 1, 1, 1, 3), (1, 1, 1, 2, 1), (1, 1, 1, 2, 2)] SHOULD number of paths 243
Number of permutations:  720 first 5 permutations: [(1, 2, 3, 4, 5, 6), (1, 2, 3, 4, 6, 5), (1, 2, 3, 5, 4, 6), (1, 2, 3, 5, 6, 4), (1, 2, 3, 6, 4, 5)] should permutations
Time to generate permutations: 0.001256 seconds
Number of full permutations:  349920 First 5: [((1, 2, 3, 4, 5, 6), (1, 1, 1, 1, 1)), ((1, 2, 3, 4, 5, 6), (1, 1, 1, 1, 2)), ((1, 2, 3, 4, 5, 6), (1, 1, 1, 1, 3)), ((1, 2, 3, 4, 5, 6), (1, 1, 1, 2, 1)), ((1, 2, 3, 4, 5, 6), (1, 1, 1, 2, 2))] should be 349920
Time to generate product: 0.049361 seconds


Step 2: simulate all paths!

In [93]:
if verbose:
    interval_time_product = time.time()

checkmark_idx = 0; check_frequency = 50; 
## path simulation
for camel_order, dice_rolls in all_full_permutations[:100]:
    ##create field copy
    field_copy = rendered_field.copy()
    
    for move_idx in range(len(dice_rolls)):
        move = dice_rolls[move_idx]
        camel = camel_order[move_idx]
        ## find camel in field_copy
        camel_idx = tuple(np.argwhere(field_copy == camel)[0])
        ## cut out stack of camel
        # Extract nonzero integers in same row and col >= camel_idx[1]
        row = camel_idx[0]
        col = camel_idx[1]
        # Extract the stack as a vector (all nonzero from (row, col) to (row, end))
        # Optimized: slice out directly using numpy for speed (all leading nonzero, so from col to first 0 or end)
        # Directly operate on the field_copy row to avoid extra reference
        end = (field_copy[row] != 0).sum()

        stack = field_copy[row, col:end].copy().reshape(1, -1)
        # Zero out in-place
        field_copy[row, col:end] = 0
        ## move camel
        below_target = False
        if camel not in [6,7]:
            target_field = row + move
            if field_copy[target_field,0] == 8:
                DO_hits[field_copy[target_field,1]-10,0] += 1
                target_field -= 1
                below_target = True
            elif field_copy[target_field,0] == 9:
                DO_hits[field_copy[target_field,1]-10,0] += 1
                target_field += 1
        else:
            target_field = row - move
            if field_copy[target_field,0] == 8:
                DO_hits[field_copy[target_field,1]-10,0] += 1
                target_field += 1
                below_target = True
            elif field_copy[target_field,0] == 9:
                DO_hits[field_copy[target_field,1]-10,0] += 1
                target_field -= 1

        ##place camel on or below target
        #print(stack,field_copy[target_field,:-stack.shape[1]].reshape(1,-1))
        if below_target:
            field_copy[target_field, :] = np.concatenate((stack, field_copy[target_field, :-stack.shape[1]].reshape(1, -1)), axis=1)
        else:
            end_target_field = (field_copy[target_field] != 0).sum()
            field_copy[target_field,end_target_field:end_target_field+stack.shape[1]] = stack
        
        if target_field > 16:
            break
    camel_positions = np.zeros((5,1))
    ## evaluation of the path
    for camel in range(1,6):
        camel_idx = tuple(np.argwhere(field_copy == camel)[0])
        row = camel_idx[0]
        col = camel_idx[1]
        camel_positions[camel-1,0] = row*7+col
    camel_ranking = np.argsort(-camel_positions.flatten())
    positions[camel_ranking[0],0] += 1
    positions[camel_ranking[1],1] += 1
    for i in range(2,5):
        positions[camel_ranking[i],2] += 1

calc_time = time.time()
print(f"Calculation time: {calc_time - interval_time_product:.6f} seconds")

end_time = time.time()
print(f"Operation time: {end_time - start_time:.6f} seconds")

Calculation time: 0.027429 seconds
Operation time: 444.974888 seconds


### put together in a numba accelerated function
Done

In [202]:
for i in range(12):
    positions, DO_hits = sim_all_moves(rendered_field, 4, n_camels_thrown, camels_not_thrown, verbose=False)
positions, DO_hits

(array([[ 47710,  59492, 242718],
        [ 64964, 100506, 184450],
        [113150,  83918, 152852],
        [ 38904,  47596, 263420],
        [ 85192,  58408, 206320]], dtype=int64),
 array([[494916],
        [     0],
        [     0],
        [     0]], dtype=int64))

In [203]:
for i in range(12):
    positions, DO_hits = sim_all_moves(rendered_field, 4, n_camels_thrown, [1,2,3,4,5], verbose=False)
rendered_field, positions, DO_hits 

(array([[ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 4,  5,  0,  0,  0,  0,  0],
        [ 1,  2,  3,  0,  0,  0,  0],
        [ 8, 10,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 7,  6,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0]]),
 array([[ 4327,  4892, 19941],
        [ 6382,  8186, 14592],
        [ 8918,  7496, 12746],
        [ 3208,  3602, 22350],
        [ 6325,  4984, 17851]], dtype=int64),
 array([[37428],
        [    0],
        [    0],
        [    0]], dtype=int64))

In [20]:
@nb.njit(cache=True)
def _all_dice_permutations(draw_n_camels):
    n_dice_rolls = 3**draw_n_camels
    dice_rolls = np.zeros((n_dice_rolls,draw_n_camels),dtype=np.int64)
    for i in range(n_dice_rolls):
        for j in range(draw_n_camels):
            dice_rolls[i,j] = (i // (3**j)) % 3 + 1
    return dice_rolls

@nb.njit(cache=True)
def _all_camel_permutations(camels_not_thrown: np.ndarray):
    """
    Generates all possible orders (permutations) to draw camels_not_thrown without replacement.
    Uses a standard algorithm since Numba does not support itertools.permutations.
    """
    n_camels_not_thrown = len(camels_not_thrown)
    out_size = 1
    for i in range(2, n_camels_not_thrown + 1):
        out_size *= i
    result = np.empty((out_size, n_camels_not_thrown), dtype=np.int64)
    a = camels_not_thrown; n=n_camels_not_thrown; 
    c = np.zeros(n, dtype=np.int64)

    result[0] = a
    idx = 1

    i = 0
    while i < n:
        if c[i] < i:
            if i % 2 == 0:
                tmp = a[0]
                a[0] = a[i]
                a[i] = tmp
            else:
                tmp = a[c[i]]
                a[c[i]] = a[i]
                a[i] = tmp

            result[idx] = a
            idx += 1

            c[i] += 1
            i = 0
        else:
            c[i] = 0
            i += 1
    return result
    
## sim_all_moves version 1:
@nb.njit(cache=True, parallel=True)
#@nb.njit(parallel=True)
def sim_all_moves_old(
    rendered_field:np.ndarray, n_players:int, n_camels_thrown:int, 
    camels_not_thrown:list[int], game_inventory_matrix:np.ndarray,
    n_threads: int = 8,
    verbose:bool = True
       ):
    '''
    This function simulates the paths for the moves of the camels
    Parameters
    ----------
    rendered_field: np.ndarray
        The rendered field, returned by the render_field function.
    n_players: int
        Number of players in the game.
    n_camels_thrown: int
        Number of camels that have been thrown.
    camels_not_thrown: list[int]
        Camels that have not been thrown. Information needed to simulate properly and to infer game version.
    game_inventory_matrix: np.ndarray
        Matrix of game inventory, calculated before the call of this function.
    verbose: bool
        Whether to print verbose output.
    Returns
    -------
    positions: array of shape (path_multiplier,5,3)
        Positions of the camels on the field.
    DO_hits: array of shape (n_players, 3)
        Hits for all desert and oasis fields.
    VOI: float
        Value of information of the move.
    '''

    len_all_camels = n_camels_thrown + len(camels_not_thrown)
    draw_n_camels = len(camels_not_thrown)
    if len_all_camels > 5:
        draw_n_camels -= 1
    # Another early exit for <=0 camels to move
    if draw_n_camels <= 0:
        if verbose:
            print("No more camels can be drawn (draw_n_camels == 0).")
        return np.zeros((5, 3), dtype=np.float64), np.zeros((n_players, 3), dtype=np.float64), 0.0

    all_dice_permutations = _all_dice_permutations(draw_n_camels)
    # Generate all possible permutations of camels_not_thrown (numba friendly: i.e., as lists of integer arrays)
    # (This is separate from all_dice_permutations, which is all product/dice rolls)
    camel_permutations = _all_camel_permutations(np.array(camels_not_thrown, dtype=np.int64))

    if verbose:
        print("draw_n_camels: ", draw_n_camels)
        print("Number of paths: ", len(all_dice_permutations), "first 5 paths: ", all_dice_permutations[:5], "SHOULD number of paths", 3**draw_n_camels)
        print("Number of permutations: ", len(camel_permutations), "first 5 permutations:", camel_permutations[:5], "should permutations")

    ## include white camel as well as black camel:
    if 6 in camels_not_thrown:
        camels_not_thrown.remove(6)
        camels_not_thrown.append(7)
        all_camel_permutations = np.concatenate((camel_permutations, _all_camel_permutations(np.array(camels_not_thrown, dtype=np.int64))), axis=0)
        camels_not_thrown.remove(7)
        camels_not_thrown.append(6)
    else:
        all_camel_permutations = camel_permutations

    if verbose:
        print("Number of full permutations: ", all_camel_permutations.shape[0]*all_dice_permutations.shape[0])

    camel_row_idx_base = np.zeros(8,dtype=np.int64)
    camel_col_idx_base = np.zeros(8,dtype=np.int64)
    if 6 in camels_not_thrown:
        for i in range(1,8):
            pos = np.argwhere(rendered_field == i)
            camel_row_idx_base[i] = pos[0][0]
            camel_col_idx_base[i] = pos[0][1]
    else:
        for i in range(1,6):
            pos = np.argwhere(rendered_field == i)
            camel_row_idx_base[i] = pos[0][0]
            camel_col_idx_base[i] = pos[0][1]

    n_perm = len(all_camel_permutations)

    if len_all_camels == 6:
        extra_dim = 3*7
    else:
        extra_dim = 3*5
    
    positions_local = np.zeros((extra_dim,n_threads, 5, 3), dtype=np.float64)
    DO_hits_local   = np.zeros((n_threads, n_players, 1), dtype=np.float64)
    #print("number of camel_orders: ", len(all_camel_permutations))
    #print("number of dice_rolls: ", len(all_dice_permutations))
    ## path simulation
    for path_idx in nb.prange(len(all_camel_permutations) * len(all_dice_permutations)):
        camel_order_idx = path_idx // len(all_dice_permutations)
        dice_idx = path_idx % len(all_dice_permutations)
        dice_rolls = all_dice_permutations[dice_idx]
    #for camel_order_idx in nb.prange(len(all_camel_permutations)):
        camel_order = all_camel_permutations[camel_order_idx]
        tid = nb.get_thread_id()
        
    #    for dice_rolls in all_dice_permutations:
            ##create field copy
        if True:
            field_copy = rendered_field.copy()

            camel_row_idx = camel_row_idx_base.copy()
            camel_col_idx = camel_col_idx_base.copy()

            for move_idx in range(len(dice_rolls)):
                move = dice_rolls[move_idx]
                camel = camel_order[move_idx]
                ## find camel in field_copy
                row = camel_row_idx[camel]
                col = camel_col_idx[camel]
                # Extract the stack as a vector (all nonzero from (row, col) to (row, end))
                # Optimized: slice out directly using numpy for speed (all leading nonzero, so from col to first 0 or end)
                # Directly operate on the field_copy row to avoid extra reference
                end = (field_copy[row] != 0).sum()

                stack = field_copy[row, col:end].copy().reshape(1, -1)
                # Zero out in-place
                field_copy[row, col:end] = 0

                ## move camel
                below_target = False
                if camel not in [6,7]:
                    target_field = row + move
                    if field_copy[target_field,0] == 8:
                        DO_hits_local[tid, field_copy[target_field,1]-10, 0] += 1
                        #local_DO_hits[field_copy[target_field,1]-10, 0] += 1
                        target_field -= 1
                        below_target = True
                    elif field_copy[target_field,0] == 9:
                        DO_hits_local[tid, field_copy[target_field,1]-10, 0] += 1
                        target_field += 1
                else:
                    target_field = row - move
                    if field_copy[target_field,0] == 8:
                        DO_hits_local[tid, field_copy[target_field,1]-10, 0] += 1
                        target_field += 1
                        below_target = True
                    elif field_copy[target_field,0] == 9:
                        DO_hits_local[tid, field_copy[target_field,1]-10, 0] += 1
                        target_field -= 1
                if target_field < 0: ## todo: implement negative target_field
                    target_field = 0
                ##place camel on or below target
                #print(stack,field_copy[target_field,:-stack.shape[1]].reshape(1,-1))
                if below_target:
                    #field_copy[target_field, :] = np.concatenate((stack, field_copy[target_field, :-stack.shape[1]].reshape(1, -1)), axis=1)
                    stack_len = stack.shape[1]
                    row_width = field_copy.shape[1]
                    end = (field_copy[target_field] != 0).sum()
                    
                    # shift existing camels right
                    for j in range(end + stack_len - 1, stack_len - 1, -1):
                        field_copy[target_field, j] = field_copy[target_field, j - stack_len]

                    # insert stack at front
                    for j in range(stack_len):
                        field_copy[target_field, j] = stack[0, j]
                else:
                    end_target_field = (field_copy[target_field] != 0).sum()
                    field_copy[target_field,end_target_field:end_target_field+stack.shape[1]] = stack

                for i in range(7):
                    if field_copy[target_field,i] == 0:
                        break
                    else:
                        camel = field_copy[target_field,i]
                        camel_row_idx[camel] = target_field
                        camel_col_idx[camel] = i                        

                if target_field > 16:
                    break
            camel_positions = np.zeros(5)
            ## evaluation of the path
            for camel in range(1,6):
                row = camel_row_idx[camel]
                col = camel_col_idx[camel]
                camel_positions[camel-1] = row*7+col
            if camel_order_idx == 0 and verbose: print(camel_positions, camel_order, dice_rolls)
            camel_ranking = np.argsort(-camel_positions.flatten())

            extra_d = 3*camel_order[0] + dice_rolls[0]-4
            
            #local_positions[extra_d, camel_ranking[0], 0] += 1
            #local_positions[extra_d, camel_ranking[1], 1] += 1
            #for i in range(2,5):
            #    local_positions[extra_d, camel_ranking[i], 2] += 1
            
            positions_local[extra_d, tid, camel_ranking[0], 0] += 1
            positions_local[extra_d, tid, camel_ranking[1], 1] += 1
            for i in range(2,5):
                positions_local[extra_d, tid, camel_ranking[i], 2] += 1

    positions = positions_local.sum(axis=1)
    DO_hits = DO_hits_local.sum(axis=0)
    #print("Number of paths: ", positions[:,0,:].sum(axis=1))

    game_inventory_multiplier = np.empty((positions.shape[0], game_inventory_matrix.shape[0], game_inventory_matrix.shape[1]))
    for i in range(positions.shape[0]):
        game_inventory_multiplier[i] = game_inventory_matrix

    ## calculate the number of paths for each first element (color+die result)
    n_paths = positions.sum(axis = 2)[:,0]
    
    VOI_array = compute_voi_array(positions)

    #print("VOI_array: ", VOI_array)
    #VOI_array[VOI_array<1] = 0 ## only count payoff elements with expected value > 1
    VOI_array = VOI_array * game_inventory_multiplier

    tmp = VOI_array.sum(axis=2)   # shape (n_paths, 3)
    next_voi = tmp.sum(axis=1)
    #print("next_voi: ", next_voi)
    
    VOI_array_now = np.zeros((5,3), dtype=np.float64)
    B = np.array([[5,3,2],[1,1,1],[-1,-1,-1]])
    positions = positions.sum(axis=0)
    inv_n = 1.0 / n_paths.sum()

    for i in nb.prange(5):
        for j in range(3):
            s = 0.0
            for k in range(3):
                s += positions[i, k] * inv_n * B[k, j]
            if s < 1:
                s = 0
            VOI_array_now[i, j] = s
    
    #print("VOI_array_now: ", VOI_array_now)
    #print("game_inventory_matrix: ", game_inventory_matrix)
    VOI_array_now = VOI_array_now*game_inventory_matrix
    now_voi  = VOI_array_now.sum()
    #del VOI_array_now
    #print("now_voi: ", now_voi)
    #print("n_paths: ", n_paths)
    VOI = ((next_voi - now_voi)*n_paths).sum()/n_paths.sum()

    return positions*inv_n, DO_hits*inv_n, VOI

In [24]:
def _camel_index_maps(rendered_field, camels_not_thrown):
    """
    Return initial row/col indices of all relevant camels.
    Not jitted; helper.
    """
    camel_row_idx_base = np.zeros(8, dtype=np.int64)
    camel_col_idx_base = np.zeros(8, dtype=np.int64)
    if 6 in camels_not_thrown:  # Extended version (7 camels)
        camel_ids = range(1, 8)
    else:
        camel_ids = range(1, 6)
    for i in camel_ids:
        pos = np.argwhere(rendered_field == i)
        camel_row_idx_base[i] = pos[0][0]
        camel_col_idx_base[i] = pos[0][1]
    return camel_row_idx_base, camel_col_idx_base

@nb.njit(cache=True, parallel=True)
def _simulate_paths(
    rendered_field, all_camel_permutations, all_dice_permutations,
    camel_row_idx_base, camel_col_idx_base,
    n_threads, n_players, extra_dim, positions_local, DO_hits_local
):
    for path_idx in nb.prange(all_camel_permutations.shape[0] * all_dice_permutations.shape[0]):
        camel_order_idx = path_idx // all_dice_permutations.shape[0]
        dice_idx = path_idx % all_dice_permutations.shape[0]
        dice_rolls = all_dice_permutations[dice_idx]
        camel_order = all_camel_permutations[camel_order_idx]
        tid = nb.get_thread_id()

        field_copy = rendered_field.copy()
        camel_row_idx = camel_row_idx_base.copy()
        camel_col_idx = camel_col_idx_base.copy()

        for move_idx in range(len(dice_rolls)):
            move = dice_rolls[move_idx]
            camel = camel_order[move_idx]
            row = camel_row_idx[camel]
            col = camel_col_idx[camel]
            end = (field_copy[row] != 0).sum()
            stack = field_copy[row, col:end].copy().reshape(1, -1)
            field_copy[row, col:end] = 0
            below_target = False

            if camel not in [6, 7]:
                target_field = row + move
                if field_copy[target_field, 0] == 8:
                    DO_hits_local[tid, field_copy[target_field, 1] - 10, 0] += 1
                    target_field -= 1
                    below_target = True
                elif field_copy[target_field, 0] == 9:
                    DO_hits_local[tid, field_copy[target_field, 1] - 10, 0] += 1
                    target_field += 1
            else:
                target_field = row - move
                if field_copy[target_field, 0] == 8:
                    DO_hits_local[tid, field_copy[target_field, 1] - 10, 0] += 1
                    target_field += 1
                    below_target = True
                elif field_copy[target_field, 0] == 9:
                    DO_hits_local[tid, field_copy[target_field, 1] - 10, 0] += 1
                    target_field -= 1
            if target_field < 0:
                target_field = 0
            if below_target:
                stack_len = stack.shape[1]
                end = (field_copy[target_field] != 0).sum()
                for j in range(end + stack_len - 1, stack_len - 1, -1):
                    field_copy[target_field, j] = field_copy[target_field, j - stack_len]
                for j in range(stack_len):
                    field_copy[target_field, j] = stack[0, j]
            else:
                end_target_field = (field_copy[target_field] != 0).sum()
                field_copy[target_field, end_target_field: end_target_field + stack.shape[1]] = stack

            for i in range(7):
                if field_copy[target_field, i] == 0:
                    break
                camel_update = field_copy[target_field, i]
                camel_row_idx[camel_update] = target_field
                camel_col_idx[camel_update] = i

            if target_field > 16:
                break
        # Ranking and counting step
        camel_positions = np.zeros(5)
        for camel_rank in range(1, 6):
            row = camel_row_idx[camel_rank]
            col = camel_col_idx[camel_rank]
            camel_positions[camel_rank - 1] = row * 7 + col
        camel_ranking = np.argsort(-camel_positions.flatten())
        extra_d = 3 * camel_order[0] + dice_rolls[0] - 4
        positions_local[extra_d, tid, camel_ranking[0], 0] += 1
        positions_local[extra_d, tid, camel_ranking[1], 1] += 1
        for i in range(2, 5):
            positions_local[extra_d, tid, camel_ranking[i], 2] += 1

@nb.njit(cache=True, parallel=True)
def _aggregate_results(
    positions_local, DO_hits_local, game_inventory_matrix, n_players
):
    positions = positions_local.sum(axis=1)
    DO_hits = DO_hits_local.sum(axis=0)
    game_inventory_multiplier = np.empty((positions.shape[0], game_inventory_matrix.shape[0], game_inventory_matrix.shape[1]))
    for i in range(positions.shape[0]):
        game_inventory_multiplier[i] = game_inventory_matrix

    n_paths = positions.sum(axis=2)[:, 0]
    VOI_array = compute_voi_array(positions)
    VOI_array = VOI_array * game_inventory_multiplier
    tmp = VOI_array.sum(axis=2)
    next_voi = tmp.sum(axis=1)
    return positions, DO_hits, n_paths, next_voi, game_inventory_multiplier

@nb.njit(cache=True, parallel=True)
def _compute_now_voi(positions, game_inventory_matrix, n_paths):
    VOI_array_now = np.zeros((5,3), dtype=np.float64)
    B = np.array([[5,3,2],[1,1,1],[-1,-1,-1]])
    agg_positions = positions.sum(axis=0)
    inv_n = 1.0 / n_paths.sum()
    for i in nb.prange(5):
        for j in range(3):
            s = 0.0
            for k in range(3):
                s += agg_positions[i, k] * inv_n * B[k, j]
            if s < 1:
                s = 0
            VOI_array_now[i, j] = s
    VOI_array_now = VOI_array_now * game_inventory_matrix
    now_voi = VOI_array_now.sum()
    agg_positions = None  # Help out numba/gc
    return now_voi

def sim_all_moves(
    rendered_field: np.ndarray,
    n_players: int,
    n_camels_thrown: int,
    camels_not_thrown: list,
    game_inventory_matrix: np.ndarray,
    n_threads: int = 8,
    verbose: bool = True,
):
    '''
    Split implementation for reduced jit compile time.
    '''
    len_all_camels = n_camels_thrown + len(camels_not_thrown)
    draw_n_camels = len(camels_not_thrown)
    if len_all_camels > 5:
        draw_n_camels -= 1
    if draw_n_camels <= 0:
        if verbose:
            print("No more camels can be drawn (draw_n_camels == 0).")
        return np.zeros((5, 3), dtype=np.float64), np.zeros((n_players, 3), dtype=np.float64), 0.0

    all_dice_permutations = _all_dice_permutations(draw_n_camels)
    camel_permutations = _all_camel_permutations(np.array(camels_not_thrown, dtype=np.int64))
    if verbose:
        print("draw_n_camels: ", draw_n_camels)
        print("Number of paths: ", len(all_dice_permutations), "first 5 paths: ", all_dice_permutations[:5], "SHOULD number of paths", 3 ** draw_n_camels)
        print("Number of permutations: ", len(camel_permutations), "first 5 permutations:", camel_permutations[:5], "should permutations")
    camels_swap_buf = list(camels_not_thrown)
    if 6 in camels_not_thrown:
        camels_swap_buf.remove(6)
        camels_swap_buf.append(7)
        cmp2 = _all_camel_permutations(np.array(camels_swap_buf, dtype=np.int64))
        all_camel_permutations = np.concatenate((camel_permutations, cmp2), axis=0)
        camels_swap_buf.remove(7)
        camels_swap_buf.append(6)
    else:
        all_camel_permutations = camel_permutations
    if verbose:
        print("Number of full permutations: ", all_camel_permutations.shape[0] * all_dice_permutations.shape[0])
    camel_row_idx_base, camel_col_idx_base = _camel_index_maps(rendered_field, camels_not_thrown)
    n_perm = len(all_camel_permutations)
    if len_all_camels == 6:
        extra_dim = 3 * 7
    else:
        extra_dim = 3 * 5
    positions_local = np.zeros((extra_dim, n_threads, 5, 3), dtype=np.float64)
    DO_hits_local = np.zeros((n_threads, n_players, 1), dtype=np.float64)
    _simulate_paths(
        rendered_field, 
        all_camel_permutations, 
        all_dice_permutations, 
        camel_row_idx_base, camel_col_idx_base,
        n_threads, n_players, extra_dim, 
        positions_local, DO_hits_local
    )
    positions, DO_hits, n_paths, next_voi, game_inventory_multiplier = _aggregate_results(
        positions_local, DO_hits_local, game_inventory_matrix, n_players)
    inv_n = 1.0 / n_paths.sum()
    now_voi = _compute_now_voi(positions, game_inventory_matrix, n_paths)
    VOI = ((next_voi - now_voi) * n_paths).sum() / n_paths.sum()
    return (positions * inv_n).sum(axis=0), DO_hits * inv_n, VOI


In [22]:
n_camels_thrown = 0
game_inventory_matrix = np.array([[1,1,2],[0,1,2],[0,0,1],[1,1,2],[0,0,0]])
#for i in range(12):
positions, DO_hits, VOI = sim_all_moves_old(rendered_field, 4, n_camels_thrown, [1, 2, 3, 4, 5, 6], game_inventory_matrix, verbose=False)
rendered_field, game_inventory_matrix, positions, DO_hits , VOI

(array([[ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 4,  5,  0,  0,  0,  0,  0],
        [ 1,  2,  3,  0,  0,  0,  0],
        [ 8, 10,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 7,  6,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0]]),
 array([[1, 1, 2],
        [0, 1, 2],
        [0, 0, 1],
        [1, 1, 2],
        [0, 0, 0]]),
 array([[0.13634545, 0.170016  , 0.69363855],
        [0.18565386, 0.28722565, 0.52712048],
        [0.32335963, 0.23982053, 0.43681984],
        [0.1111797 , 0.13601966, 0.752800

In [25]:
positions, DO_hits, VOI = sim_all_moves(rendered_field, 4, n_camels_thrown, [1, 2, 3, 4, 5, 6], game_inventory_matrix, verbose=False)
rendered_field, game_inventory_matrix, positions, DO_hits , VOI

(array([[ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 4,  5,  0,  0,  0,  0,  0],
        [ 1,  2,  3,  0,  0,  0,  0],
        [ 8, 10,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 7,  6,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0]]),
 array([[1, 1, 2],
        [0, 1, 2],
        [0, 0, 1],
        [1, 1, 2],
        [0, 0, 0]]),
 array([[0.13634545, 0.170016  , 0.69363855],
        [0.18565386, 0.28722565, 0.52712048],
        [0.32335963, 0.23982053, 0.43681984],
        [0.1111797 , 0.13601966, 0.752800

In [26]:
np.array([[0.13634545, 0.170016  , 0.69363855],
        [0.18565386, 0.28722565, 0.52712048],
        [0.32335963, 0.23982053, 0.43681984],
        [0.1111797 , 0.13601966, 0.75280064],
        [0.24346136, 0.16691815, 0.58962048]])-np.array([[0.13634545, 0.170016  , 0.69363855],
        [0.18565386, 0.28722565, 0.52712048],
        [0.32335963, 0.23982053, 0.43681984],
        [0.1111797 , 0.13601966, 0.75280064],
        [0.24346136, 0.16691815, 0.58962048]])

array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]])

bla

In [223]:
len("════════════════════════                                                                    ════════════════════════")

116

In [220]:
[i[24:-24] for i in """
║      xyz----zyx      ║                                                                    ║      xyz----zyx      ║
║      xyz----zyx      ║             <16>               < 1>               < 2>             ║      xyz----zyx      ║
║      xyz----zyx      ║     <15>                                                  < 3>     ║      xyz----zyx      ║
║      xyz----zyx      ║                                                                    ║      xyz----zyx      ║
║      xyz----zyx      ║                                                                    ║      xyz----zyx      ║
║      xyz----zyx      ║   <14>                                                      < 4>   ║      xyz----zyx      ║
║      xyz----zyx      ║                                                                    ║      xyz----zyx      ║
════════════════════════                                                                    ════════════════════════
║      xyz----zyx      ║                                                                    ║      xyz----zyx      ║
║      xyz----zyx      ║                                                                    ║      xyz----zyx      ║
║      xyz----zyx      ║                 ═════════════════════════════════                  ║      xyz----zyx      ║
║      xyz----zyx      ║   <13>          ║Value of Information:     {VOI:5.3f}║           < 5>   ║      xyz----zyx      ║
║      xyz----zyx      ║                 ═════════════════════════════════                  ║      xyz----zyx      ║
║      xyz----zyx      ║                                                                    ║      xyz----zyx      ║
║      xyz----zyx      ║                                                                    ║      xyz----zyx      ║
════════════════════════                                                                    ════════════════════════
║      xyz----zyx      ║                                                                    ║      xyz----zyx      ║
║      xyz----zyx      ║   <12>                                                      < 6>   ║      xyz----zyx      ║
║      xyz----zyx      ║                                                                    ║      xyz----zyx      ║
║      xyz----zyx      ║                                                                    ║      xyz----zyx      ║
║      xyz----zyx      ║     <11>                                                  < 7>     ║      xyz----zyx      ║
║      xyz----zyx      ║             <10>               < 9>               < 8>             ║      xyz----zyx      ║
║      xyz----zyx      ║                                                                    ║      xyz----zyx      ║""".split("\n")]

['',
 '                                                                    ',
 '             <16>               < 1>               < 2>             ',
 '     <15>                                                  < 3>     ',
 '                                                                    ',
 '                                                                    ',
 '   <14>                                                      < 4>   ',
 '                                                                    ',
 '                                                                    ',
 '                                                                    ',
 '                                                                    ',
 '                 ═════════════════════════════════                  ',
 '   <13>          ║Value of Information:     {VOI:5.3f}║           < 5>   ',
 '                 ═════════════════════════════════                  ',
 '                                       

VOI calculation blueprint

In [8]:
import numpy as np
game_inventory_matrix = np.array([[1,1,2],[0,0,1],[0,1,2],[0,0,2],[1,1,2]])
path_multiplier = 21

positions = np.ones((path_multiplier,5,3))
positions[:-6,:,:]*=2
##create a 3D array of the game inventory matrix, repeated for each paths first element
game_inventory_multiplier = np.tile(game_inventory_matrix, (positions.shape[0],1,1))

## calculate the number of paths for each first element (color+die result)
n_paths = positions.sum(axis = 2)[:,0]

#print(n_paths)
VOI_array = np.matmul(
    positions * (1/n_paths)[:, np.newaxis, np.newaxis], 
    np.tile([[5,3,2],[1,1,1],[-1,-1,-1]], (positions.shape[0],1)).reshape(positions.shape[0],3,3))
VOI_array[VOI_array<1] = 0
VOI_array = VOI_array * game_inventory_multiplier

#print(VOI_array)

next_voi = VOI_array.sum(axis = (1,2))
#del VOI_array

VOI_array_now  = np.matmul(
    (positions).sum(axis = 0)/n_paths.sum(),
    np.array([[5,3,2],[1,1,1],[-1,-1,-1]]))
#print(VOI_array_now)
VOI_array_now[VOI_array_now<1] = 0
VOI_array_now*=game_inventory_matrix
#print(VOI_array_now)
now_voi  = VOI_array_now.sum()
print(now_voi)
print(next_voi)
VOI = ((next_voi - now_voi)*n_paths).sum()/n_paths.sum()
print(VOI)

6.333333333333333
[6.33333333 6.33333333 6.33333333 6.33333333 6.33333333 6.33333333
 6.33333333 6.33333333 6.33333333 6.33333333 6.33333333 6.33333333
 6.33333333 6.33333333 6.33333333 6.33333333 6.33333333 6.33333333
 6.33333333 6.33333333 6.33333333]
0.0


In [ ]:
@nb.njit
def compute_voi_array(positions):
    n = positions.shape[0]
    VOI = np.zeros((n, 5, 3), dtype=np.float64)

    score = np.array([[5.0, 3.0, 2.0],
                      [1.0, 1.0, 1.0],
                      [-1.0, -1.0, -1.0]])

    for i in nb.prange(n):
        n_paths = positions[i, :, :].sum()
        if n_paths == 0:
            continue

        inv_n = 1.0 / n_paths

        for r in range(5):
            for c in range(3):
                s = 0.0
                for k in range(3):
                    s += positions[i, r, k] * inv_n * score[k, c]
                if s < 1:
                    s = 0
                VOI[i, r, c] = s

    return VOI

game_inventory_multiplier = np.empty((positions.shape[0], game_inventory_matrix.shape[0], game_inventory_matrix.shape[1]))
for i in range(positions.shape[0]):
    game_inventory_multiplier[i] = game_inventory_matrix

## calculate the number of paths for each first element (color+die result)
n_paths = positions.sum(axis = 2)[:,0]

VOI_array = compute_voi_array(positions)

#VOI_array[VOI_array<1] = 0 ## only count payoff elements with expected value > 1
VOI_array = VOI_array * game_inventory_multiplier

tmp = VOI_array.sum(axis=2)   # shape (n_paths, 3)
next_voi = tmp.sum(axis=1)

VOI_array_now = np.zeros((5,3), dtype=np.float64)
B = np.array([[5,3,2],[1,1,1],[-1,-1,-1]])
positions = positions.sum(axis=0)
inv_n = 1.0 / n_paths.sum()

for i in nb.prange(5):
    for j in range(3):
        s = 0.0
        for k in range(3):
            s += positions[i, k] * inv_n * B[k, j]
        if s < 1:
            s = 0
        VOI_array_now[i, j] = s

VOI_array_now = VOI_array_now*game_inventory_matrix
now_voi  = VOI_array_now.sum()
#del VOI_array_now

VOI = ((next_voi - now_voi)*n_paths).sum()/n_paths.sum()

#DO_hits   = DO_hits_local.sum(axis=0)
DO_hits   = DO_hits_acc#.sum(axis=0)



In [9]:
VOI_array.shape

(21, 5, 3)

### Numba accelerated function tester

In [6]:
### Numba accelerated function tester
n_camels_thrown = 0
game_inventory_matrix = np.array([[1,1,2],[0,1,2],[0,0,1],[1,1,2],[0,0,0]])
#for i in range(12):
positions, DO_hits, VOI = sim_all_moves(rendered_field, 4, n_camels_thrown, [1, 2, 3, 4, 5, 6], game_inventory_matrix, verbose=False)
rendered_field, game_inventory_matrix, positions, DO_hits , VOI

(array([[ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 4,  5,  0,  0,  0,  0,  0],
        [ 1,  2,  3,  0,  0,  0,  0],
        [ 8, 10,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 7,  6,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0]]),
 array([[1, 1, 2],
        [0, 1, 2],
        [0, 0, 1],
        [1, 1, 2],
        [0, 0, 0]]),
 array([[0.13634545, 0.170016  , 0.69363855],
        [0.18565386, 0.28722565, 0.52712048],
        [0.32335963, 0.23982053, 0.43681984],
        [0.1111797 , 0.13601966, 0.752800

In [24]:
sum([113150.*5,  83918.*1, 152852.*-1])/349920

1.4197988111568358

In [39]:
for i in range(12):
    positions, DO_hits, VOI = sim_all_moves(rendered_field, 4, n_camels_thrown, [1, 2, 3, 4, 5, 6], game_inventory_matrix, verbose=False)
rendered_field, positions, DO_hits , VOI

(array([[ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 4,  5,  0,  0,  0,  0,  0],
        [ 1,  2,  3,  0,  0,  0,  0],
        [ 8, 10,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 7,  6,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0]]),
 array([[0.13634545, 0.170016  , 0.69363855],
        [0.18565386, 0.28722565, 0.52712048],
        [0.32335963, 0.23982053, 0.43681984],
        [0.1111797 , 0.13601966, 0.75280064],
        [0.24346136, 0.16691815, 0.58962048]]),
 array([[1.414369],
        [0.      ],
    

In [12]:
sum([ 82939.,  56964., 200297.])

340200.0

In [7]:
for i in range(12):
    positions, DO_hits, VOI = sim_all_moves(rendered_field, 4, n_camels_thrown, [1, 2, 3, 4, 5], game_inventory_matrix, verbose=False)
rendered_field, positions, DO_hits , VOI

(array([[ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 4,  5,  0,  0,  0,  0,  0],
        [ 1,  2,  3,  0,  0,  0,  0],
        [ 8, 10,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 7,  6,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0],
        [ 0,  0,  0,  0,  0,  0,  0]]),
 array([[0.1483882 , 0.16776406, 0.68384774],
        [0.21886145, 0.28072702, 0.50041152],
        [0.3058299 , 0.25706447, 0.43710562],
        [0.11001372, 0.12352538, 0.76646091],
        [0.21690672, 0.17091907, 0.61217421]]),
 array([[1.28353909],
        [0.        ],


In [ ]:
"Probabilities  White   Green   Blue    Yellow  Red   "
"First          40.00%  20.00%  20.00%  10.00%  10.00%"
"Second         20.00%  30.00%  30.00%  10.00%  10.00%"
"First          40.00%  50.00%  50.00%  80.00%  80.00%"

"Payoffs        White   Green   Blue    Yellow  Red  "
"5-Plate         1.80    0.80    0.80   -0.20   -0.20"
"3-Plate         1.00    0.40    0.40   -0.40   -0.40"
"2-Plate         0.60    0.20    0.20   -0.50   -0.50"
"2-Plate         0.60    0.20    0.20   -0.50   -0.50"

array([3, 3, 3, 3, 3])

In [43]:
np.matmul(
    np.array([
        [.4,.2,.4],
        [.2,.3,.5],
        [.2,.3,.5],
        [.1,.1,.8],
        [.1,.1,.8],    
    ]),
    np.array([[5,3,2],[1,1,1],[-1,-1,-1]])
)

array([[ 1.8,  1. ,  0.6],
       [ 0.8,  0.4,  0.2],
       [ 0.8,  0.4,  0.2],
       [-0.2, -0.4, -0.5],
       [-0.2, -0.4, -0.5]])

In [73]:
def format_tables(probabilities: pd.DataFrame, payoffs: pd.DataFrame, game_inventory_list: list = [],extended: bool = False) -> str:
    # layout parameters
    row_label_width = 14
    col_width = 10   # includes the extra leading space
    camels = probabilities.index.tolist()

    lines_probabilities = []

    # ---------- Probabilities ----------
    header = (
        f"{'Probabilities':<{row_label_width}}"
        + "".join(f"{c:>{col_width-1}} " for c in camels)
    )
    lines_probabilities.append(header)

    for row in probabilities.columns:
        line = f"{row:<{row_label_width}}"
        for c in camels:
            val = probabilities.loc[c, row] * 100.0
            line += f"{val:>{col_width - 1}.2f}%"
        lines_probabilities.append(line)

    # ---------- Payoffs ----------
    lines_payoffs = []
    header = (
        f"{'Payoffs':<{row_label_width}}"
        + "".join(f"{c:>{col_width-1}} " for c in camels)
    )
    lines_payoffs.append(header)
    line_idx = 0
    for row in payoffs.columns:
        line = f"{row:<{row_label_width}}"
        for c in camels:
            val = payoffs.loc[c, row]
            # signed, aligned float
            plate_name = f"{c} [{row[0]}]"
            plate_there = plate_name in game_inventory_list
            if plate_there and row[0] == "2" and extended:
                if game_inventory_list.count(plate_name) == 1:
                    plate_there = False
            if not plate_there:
                line += f"\033[31m{val:>{col_width-1}.2f}\033[0m "
            else:
                line += f"{val:>{col_width-1}.2f} "
        lines_payoffs.append(line)
        line_idx += 1
    if extended:
        line = f"{row:<{row_label_width}}"
        for c in camels:
            val = payoffs.loc[c, row]
            # signed, aligned float
            plate_name = f"{c} [{row[0]}]"
            if plate_name not in game_inventory_list:
                line += f"\033[31m{val:>{col_width-1}.2f}\033[0m "
            else:
                line += f"{val:>{col_width-1}.2f} "
        lines_payoffs.append(line)

    return lines_probabilities, lines_payoffs

In [74]:
probabilities = pd.DataFrame(
    np.array([ 
        [.4,.2,.4], [.2,.3,.5], [.2,.3,.5], [.1,.1,.8], [.1,.1,.8], ]), 
        index = ["White", "Green", "Blue", "Yellow", "Red"], 
        columns = ["First", "Second", "Lose"]) 
payoffs = pd.DataFrame(
    np.array([ [ 1.8, 1. , 0.6], [ 0.8, 0.4, 0.2], [ 0.8, 0.4, 0.2], [-0.2, -0.4, -0.5], [-0.2, -0.4, -0.5] ]),
    index = ["White", "Green", "Blue", "Yellow", "Red"], 
    columns = ["5-Plate", "3-Plate", "2-Plate"])

prob_str, payoffs_str = format_tables(probabilities, payoffs, ["Yellow [5]", "Yellow [2]", "Yellow [2]", "Yellow [3]", "Blue [2]"],True)
print("\n".join(prob_str))
print("")
print("\n".join(payoffs_str))

Probabilities     White     Green      Blue    Yellow       Red 
First             40.00%    20.00%    20.00%    10.00%    10.00%
Second            20.00%    30.00%    30.00%    10.00%    10.00%
Lose              40.00%    50.00%    50.00%    80.00%    80.00%

Payoffs           White     Green      Blue    Yellow       Red 
5-Plate            1.80      0.80      0.80     -0.20     -0.20 
3-Plate            1.00      0.40      0.40     -0.40     -0.40 
2-Plate            0.60      0.20      0.20     -0.50     -0.50 
2-Plate            0.60      0.20      0.20     -0.50     -0.50 


### Currently unused test cases

In [ ]:
                    

demo3_players = {"Michael":player("Micheal"),"Dwight":player("Dwight"),"Jim":player("Jim")}

demoE = Field([[], [], [], [], [], [], [], [], [], ['White'], ['Blue'], [], ['Orange'], [],
               ['Yellow', 'Green'], [], [], [], []],copy.deepcopy(demo_players))

demo1 = Field([[], [], [], [], [], ['White','Blue','Orange'], ['DESERT', 'Player 4'], ['Yellow', 'Green'], 
               ['DESERT', 'Player 1'], [], [], [], [], [], [], [], [], [], []],copy.deepcopy(demo_players))

demoS2 = Field([['White','Blue'], ['Yellow', 'Green'], ['Orange'], ['DESERT', 'Player 1'], [],
               ['DESERT', 'Player 4'], [], [], [], [], [], [], [], [], [], [], [], [], []],copy.deepcopy(demo_players))

demo3 = Field([['White','Blue'], ['Yellow', 'Green'], ['Orange'], [], [], [], [], [], 
               [], [], [], [], [], [], [], [], [], [], []],copy.deepcopy(demo3_players))

demoS = Field([['White','Blue'], ['Yellow', 'Green'], ['Orange'], [], [], [], [], [], 
               [], [], [], [], [], [], [], [], [], [], []],copy.deepcopy(demo_players))


CC1 = CamelUp(4,field = demo1)
CC = CamelUp(4,field = demoE)
CCS = CamelUp(4,field = demoS)
CC3 = CamelUp(4,field = demo3)
CCS2 = CamelUp(4,field = demoS2)


In [ ]:
start_scenarios = {
    [5,0,0]:1, [0,5,0]:1, [0,0,5]:1,
    [4,1,0]:5, [4,0,1]:5, [1,4,0]:5, [0,4,1]:5, [1,0,4]:5, [0,1,4]:5,
    [3,1,1]:20, [1,3,1]:20, [1,1,3]:20,
    [3,2,0]:10, [3,0,2]:10, [2,3,0]:10, [0,3,2]:10, [2,0,3]:10, [0,3,2]:10,
    [2,2,1]:30, [2,1,2]:30, [1,2,2]:30}